# nb0 — Phase 0: Chuẩn bị dữ liệu (VSEC + bộ test tùy chọn)

Notebook đầu tiên trong chuỗi nb0→nb3 (DESIGN.md §7): load VSEC từ Hugging Face, chuẩn hóa NFC, sanity-check annotation vị trí, dedupe (trong VSEC + chéo với bộ test tự thu **nếu có**), chia train/val 90/10 stratified theo `error_count` (seed 42), xuất file trung gian cho nb1.

Tham chiếu: `PROJECT.md` §3 (schema VSEC), §5 (chống leakage) · `DESIGN.md` §1 (quyết định đã chốt), §2 (bộ test), §9 (checklist leakage).

**Trên Kaggle**: bật Internet trước khi chạy (*Settings → Environment → Internet → ON*) để `pip install` và tải dataset từ Hugging Face.

## Checklist chống leakage của notebook này (DESIGN.md §9)

- Dedupe exact + near-dup **trong** VSEC và **chéo** VSEC↔test **trước khi** chia tập.
- Chia **theo câu** (không theo âm tiết), stratified theo `error_count` binned 1/2/3/≥4, seed 42 cố định, log rõ tỷ lệ + phương pháp.
- Không dùng nhãn val/test để fit bất kỳ thống kê nào; nb0 không làm augmentation (nếu có, chỉ áp dụng trên train ở notebook sau).
- Mọi số câu bị loại ở từng bước đều log vào `manifest.json` để tái lập.

## 0. Cấu hình — mọi tham số gom một chỗ (DESIGN.md §7)

| Tham số | Giá trị | Ý nghĩa |
|---|---|---|
| `SEED` | `42` | seed cố định cho chia tập (tái lập được) |
| `VAL_SIZE` | `0.1` | VSEC chia train/val **90/10** |
| `NEARDUP_THRESHOLD` | `92` | ngưỡng near-duplicate, `rapidfuzz.token_set_ratio` (0–100) |
| `CROSS_DUP_KEEP` | `"test"` | trùng chéo VSEC↔test: **giữ phía test, loại khỏi VSEC** (tránh leakage vào train/val) |
| `ERROR_BINS` | `[1, 2, 3, 4]` | bin `error_count` để stratify: 1/2/3/≥4 (VSEC 100% câu có lỗi → không có bin 0) |

Đường dẫn: input quét `/kaggle/input` (Kaggle) hoặc `./data` (local) để **tự dò** file test tùy chọn (`.csv`/`.json`/`.jsonl` có cột/khóa `text` + `corrected_text`); output xuống `/kaggle/working` (Kaggle) hoặc `./out` (local).

In [ ]:
import datetime
import json
from pathlib import Path

SEED = 42
VAL_SIZE = 0.1
NEARDUP_THRESHOLD = 92
CROSS_DUP_KEEP = "test"
ERROR_BINS = [1, 2, 3, 4]

HF_DATASET_ID = "nguyenthanhasia/vsec-vietnamese-spell-correction"

INPUT_DIRS = [Path(p) for p in ["/kaggle/input", "./data"] if Path(p).is_dir()]
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("./out")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.datetime.now().isoformat(timespec="seconds")

print("INPUT_DIRS :", [str(p) for p in INPUT_DIRS] or "(không có — sẽ bỏ qua phần test tùy chọn)")
print("OUTPUT_DIR :", OUTPUT_DIR)
print(f"SEED={SEED}  VAL_SIZE={VAL_SIZE}  NEARDUP_THRESHOLD={NEARDUP_THRESHOLD}  CROSS_DUP_KEEP={CROSS_DUP_KEEP!r}")

## 1. Cài thư viện

`datasets` (tải VSEC từ HF hub) và `rapidfuzz` (near-duplicate). Trên Kaggle đã cài sẵn một phần — cell này chỉ cài khi thiếu, idempotent. **Cần Internet ON.**

In [ ]:
import importlib.util
import subprocess
import sys

for pkg in ["datasets", "rapidfuzz"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("datasets + rapidfuzz sẵn sàng")

## 2. Hàm dùng chung

Đặt đầu notebook, **copy nhất quán giữa nb0→nb3** (DESIGN.md §7):

- `nfc_normalize(s)` — Unicode **NFC** + strip + gộp khoảng trắng (web tiếng Việt thường dính NFD; DESIGN.md §2.4).
- `syllable_tokenize(s)` — NFC + tách theo whitespace: khớp quy ước VSEC tách dấu câu bằng space (`"sanh , sạch , đẹp"`), không cần tokenizer ngoài.
- `exact_dedupe(texts_norm)` — trả `(keep_idx, drop_idx)` giữ occurrence đầu tiên.
- `near_dedupe(texts_norm, keep_idx, threshold)` — greedy giữ câu xuất hiện trước, so cặp bằng `rapidfuzz.process.cdist` (`token_set_ratio`, `score_cutoff`), chỉ xét trong danh sách `keep_idx`.

In [ ]:
import re
import unicodedata

from rapidfuzz import fuzz
from rapidfuzz import process as rf_process

_WS_RE = re.compile(r"\s+")


def nfc_normalize(s):
    return _WS_RE.sub(" ", unicodedata.normalize("NFC", s)).strip()


def syllable_tokenize(s):
    return nfc_normalize(s).split()


def exact_dedupe(texts_norm):
    seen = set()
    keep, drop = [], []
    for i, t in enumerate(texts_norm):
        if t in seen:
            drop.append(i)
        else:
            seen.add(t)
            keep.append(i)
    return keep, drop


def near_dedupe(texts_norm, keep_idx, threshold):
    if not keep_idx:
        return [], []
    sub = [texts_norm[i] for i in keep_idx]
    sim = rf_process.cdist(sub, sub, scorer=fuzz.token_set_ratio, score_cutoff=threshold, workers=-1)
    kept_pos, drop = [], []
    for j in range(len(sub)):
        if kept_pos and sim[j, kept_pos].max() >= threshold:
            drop.append(keep_idx[j])
        else:
            kept_pos.append(j)
    return [keep_idx[j] for j in kept_pos], drop

## 3. Load VSEC từ Hugging Face

Dataset chỉ có **1 split `train`** (PROJECT.md §3) — nb0 tự chia train/val ở bước 7. In schema + thống kê mô tả.

In [ ]:
import collections

from datasets import load_dataset

ds = load_dataset(HF_DATASET_ID, split="train")

print(ds)
err_counter = collections.Counter(ds["error_count"])
print("Phân bố error_count :", dict(sorted(err_counter.items())))
print("has_errors luôn true:", all(ds["has_errors"]))
print("Tổng số lỗi         :", sum(ds["error_count"]))
print()
print("Ví dụ dòng đầu (cắt 1500 ký tự):")
print(json.dumps(ds[0], ensure_ascii=False, indent=2)[:1500])

## 4. Chuẩn hóa NFC + sanity-check annotation vị trí

Đối chiếu `syllable_annotations[].position` với index token sau khi tokenize. Các edge case đã biết (PROJECT.md §3.2): âm tiết tách sai (`"Tuy nh iên"`), dính từ (`"vàahệ"`), chèn ký tự thừa (`"aNăng"`) → position có thể lệch khỏi token index.

**Không fail cứng** — chỉ đếm, log và in tối đa 5 mẫu để đối chiếu về sau (DESIGN.md §6: bookkeeping vị trí khi có chèn/xóa/merge/split là chỗ dễ sai nhất, sẽ xử lý kỹ ở nb1 align).

In [ ]:
texts = [nfc_normalize(t) for t in ds["text"]]
corrected = [nfc_normalize(t) for t in ds["corrected_text"]]

n_mismatch_rows = 0
n_mismatch_ann = 0
mismatch_samples = []

for i in range(len(ds)):
    tokens = syllable_tokenize(texts[i])
    row_bad = False
    for ann in ds[i]["syllable_annotations"]:
        p = ann["position"]
        expected = nfc_normalize(ann["syllable"])
        ok = isinstance(p, int) and 0 <= p < len(tokens) and tokens[p] == expected
        if not ok:
            row_bad = True
            n_mismatch_ann += 1
            if len(mismatch_samples) < 5:
                got = tokens[p] if isinstance(p, int) and 0 <= p < len(tokens) else "<out of range>"
                mismatch_samples.append((i, p, ann["syllable"], got))
    if row_bad:
        n_mismatch_rows += 1

print(f"Câu có ít nhất 1 annotation lệch vị trí : {n_mismatch_rows}/{len(ds)}")
print(f"Tổng annotation lệch vị trí             : {n_mismatch_ann}")
print("Mẫu (row_id, position, annotation, token_tai_vi_tri):")
for s in mismatch_samples:
    print("  ", s)

## 5. Dedupe trong VSEC (exact → near-dup)

- **Exact**: trên `text` đã NFC-normalize, giữ occurrence đầu tiên.
- **Near-dup**: `rapidfuzz.process.cdist` (`token_set_ratio`, `score_cutoff = NEARDUP_THRESHOLD`), tham lam giữ câu xuất hiện trước.

Ma trận 9,3k² ≈ 87M cặp — `cdist` + `score_cutoff` + `workers=-1` chạy trong vài phút trên CPU Kaggle. Nếu quá chậm có thể pre-bucket theo độ dài (chưa cần làm trước).

In [ ]:
vsec_keep_exact, vsec_drop_exact = exact_dedupe(texts)
print(f"Exact-dedupe : loại {len(vsec_drop_exact)} câu, còn {len(vsec_keep_exact)}")

vsec_within, vsec_drop_near = near_dedupe(texts, vsec_keep_exact, NEARDUP_THRESHOLD)
print(f"Near-dedupe  : loại {len(vsec_drop_near)} câu, còn {len(vsec_within)}")

## 6. (Tùy chọn) Bộ test 6k — tự dò + dedupe trong test + dedupe chéo VSEC↔test

Bộ test 6k tự thu thập **chưa tồn tại** ở thời điểm nb0 → phần này **tự dò** file `.csv`/`.json`/`.jsonl` có cột/khóa `text` + `corrected_text` trong input dir (bỏ qua file có tên chứa `vsec` để tránh nhận nhầm dataset VSEC nếu được mount lên input).

- **Có file** → NFC normalize, dedupe trong test (exact + near), rồi dedupe chéo VSEC↔test: trùng thì **giữ phía test, loại khỏi VSEC** (`CROSS_DUP_KEEP = "test"`) — rủi ro trùng cao nhất vì cùng nguồn báo chí/giáo dục (DESIGN.md §2).
- **Không có file** → in cảnh báo, bỏ qua dedupe trong test + dedupe chéo, notebook vẫn chạy trọn vẹn với VSEC.

In [ ]:
def find_test_candidates(input_dirs):
    candidates = []
    for d in input_dirs:
        for p in sorted(d.rglob("*")):
            if p.is_file() and p.suffix.lower() in {".csv", ".json", ".jsonl"} and "vsec" not in p.name.lower():
                candidates.append(p)
    return candidates


def load_test_records(path):
    if path.suffix.lower() == ".csv":
        import pandas as pd
        df = pd.read_csv(path)
        cols = {str(c).strip().lower(): c for c in df.columns}
        if not {"text", "corrected_text"} <= set(cols):
            return None
        df = df.rename(columns={cols["text"]: "text", cols["corrected_text"]: "corrected_text"})
        return [
            {"text": str(t), "corrected_text": str(c)}
            for t, c in zip(df["text"], df["corrected_text"])
        ]
    raw = path.read_text(encoding="utf-8").strip()
    try:
        data = json.loads(raw)
        if isinstance(data, dict):
            data = data.get("data", data.get("test", []))
        if (
            isinstance(data, list)
            and data
            and all(isinstance(r, dict) and "text" in r and "corrected_text" in r for r in data)
        ):
            return [{"text": str(r["text"]), "corrected_text": str(r["corrected_text"])} for r in data]
    except json.JSONDecodeError:
        pass
    records = []
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            return None
        if not (isinstance(r, dict) and "text" in r and "corrected_text" in r):
            return None
        records.append({"text": str(r["text"]), "corrected_text": str(r["corrected_text"])})
    return records or None


test_path = None
test_records = None
for p in find_test_candidates(INPUT_DIRS):
    loaded = load_test_records(p)
    if loaded:
        test_path, test_records = p, loaded
        break

TEST_FOUND = test_records is not None
if TEST_FOUND:
    print("Tìm thấy file test:", test_path, "→", len(test_records), "câu")
else:
    print("[Cảnh báo] Chưa có bộ test — không tìm thấy file có 'text' + 'corrected_text' trong:",
          [str(d) for d in INPUT_DIRS] or "(không có input dir)")
    print("→ Bỏ qua dedupe trong test + dedupe chéo VSEC↔test; notebook vẫn chạy trọn vẹn với VSEC.")

In [ ]:
cross_drop_exact = 0
cross_drop_near = 0
vsec_final = list(vsec_within)
test_export = []

if TEST_FOUND:
    test_raw_n = len(test_records)
    test_texts = [nfc_normalize(r["text"]) for r in test_records]
    test_corrected = [nfc_normalize(r["corrected_text"]) for r in test_records]

    test_keep_exact_idx, test_drop_exact_idx = exact_dedupe(test_texts)
    test_drop_exact = len(test_drop_exact_idx)
    print(f"Test exact-dedupe : loại {test_drop_exact}, còn {len(test_keep_exact_idx)}")

    test_keep, test_drop_near_idx = near_dedupe(test_texts, test_keep_exact_idx, NEARDUP_THRESHOLD)
    test_drop_near = len(test_drop_near_idx)
    print(f"Test near-dedupe  : loại {test_drop_near}, còn {len(test_keep)}")

    if CROSS_DUP_KEEP != "test":
        raise ValueError(f"CROSS_DUP_KEEP chưa hỗ trợ: {CROSS_DUP_KEEP}")

    kept_test_texts = [test_texts[j] for j in test_keep]
    kept_test_set = set(kept_test_texts)
    cross_exact_idx = [i for i in vsec_final if texts[i] in kept_test_set]
    remaining = [i for i in vsec_final if texts[i] not in kept_test_set]
    if kept_test_texts and remaining:
        sim = rf_process.cdist(
            [texts[i] for i in remaining], kept_test_texts,
            scorer=fuzz.token_set_ratio, score_cutoff=NEARDUP_THRESHOLD, workers=-1,
        )
        cross_near_idx = [remaining[j] for j in range(len(remaining)) if sim[j].max() >= NEARDUP_THRESHOLD]
    else:
        cross_near_idx = []
    cross_drop_exact, cross_drop_near = len(cross_exact_idx), len(cross_near_idx)
    cross_drop = set(cross_exact_idx) | set(cross_near_idx)
    vsec_final = [i for i in vsec_final if i not in cross_drop]

    test_export = [
        {
            "text_raw": test_records[j]["text"],
            "text": test_texts[j],
            "corrected_text_raw": test_records[j]["corrected_text"],
            "corrected_text": test_corrected[j],
        }
        for j in test_keep
    ]

    print(f"Cross-dedupe      : loại khỏi VSEC {cross_drop_exact} exact + {cross_drop_near} near, còn {len(vsec_final)}")
    print(f"Test sau dedupe   : {len(test_export)} câu")
else:
    test_raw_n = test_drop_exact = test_drop_near = 0
    print(f"VSEC sau dedupe trong = sau dedupe chéo: {len(vsec_final)}")

## 7. Chia train/val 90/10 — stratified theo `error_count` binned, seed 42

Chia **theo câu**, thực hiện **sau khi** mọi bước dedupe xong (DESIGN.md §9). Bin `error_count`: 1/2/3/≥4 — `has_errors` luôn `true` nên mọi câu có ≥1 lỗi, không có bin 0. Dùng `sklearn.train_test_split(stratify=...)`, `random_state = SEED` để tái lập được.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

error_counts = ds["error_count"]
max_bin = max(ERROR_BINS)
bins = [min(error_counts[i], max_bin) for i in vsec_final]

train_pos, val_pos = train_test_split(
    list(range(len(vsec_final))), test_size=VAL_SIZE, random_state=SEED, stratify=bins
)
train_idx = [vsec_final[j] for j in train_pos]
val_idx = [vsec_final[j] for j in val_pos]

bin_names = {b: (str(b) if b < max_bin else f">={b}") for b in ERROR_BINS}
dist_rows = []
for b in ERROR_BINS:
    dist_rows.append((
        bin_names[b],
        sum(1 for x in bins if x == b),
        sum(1 for j in train_pos if bins[j] == b),
        sum(1 for j in val_pos if bins[j] == b),
    ))
print(pd.DataFrame(dist_rows, columns=["error_bin", "sau_dedupe", "train", "val"]).to_string(index=False))
print(f"\ntrain = {len(train_idx)} câu, val = {len(val_idx)} câu (tỉ lệ {1 - VAL_SIZE:.0%}/{VAL_SIZE:.0%}, seed {SEED})")

## 8. Báo cáo flow + kiểm chứng invariants + xuất file

Xuất xuống output dir (`/kaggle/working` hoặc `./out`):

- `vsec_train.jsonl`, `vsec_val.jsonl` — bản ghi VSEC đầy đủ (giữ cả `text_raw` gốc và bản NFC), gắn `row_id` về index gốc trên HF để truy vết.
- `test_normalized.jsonl` — chỉ khi có bộ test.
- `manifest.json` — seed, tỷ lệ, ngưỡng dedupe, số câu loại từng bước, số mismatch position, ngày tạo — nb1 đọc file này trước để bảo đảm nhất quán.

In [ ]:
def vsec_record(i, split):
    row = ds[i]
    return {
        "row_id": i,
        "split": split,
        "text_raw": row["text"],
        "text": texts[i],
        "corrected_text_raw": row["corrected_text"],
        "corrected_text": corrected[i],
        "error_count": row["error_count"],
        "error_positions": row["error_positions"],
        "has_errors": row["has_errors"],
        "syllable_annotations": row["syllable_annotations"],
        "correction_pairs": row["correction_pairs"],
    }


def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


train_records = [vsec_record(i, "train") for i in train_idx]
val_records = [vsec_record(i, "val") for i in val_idx]
write_jsonl(OUTPUT_DIR / "vsec_train.jsonl", train_records)
write_jsonl(OUTPUT_DIR / "vsec_val.jsonl", val_records)
if TEST_FOUND:
    write_jsonl(OUTPUT_DIR / "test_normalized.jsonl", test_export)
print("Đã ghi jsonl vào", OUTPUT_DIR)

In [ ]:
flow_rows = [
    ("VSEC raw", len(ds)),
    ("sau exact-dedupe", len(vsec_keep_exact)),
    ("sau near-dedupe", len(vsec_within)),
]
if TEST_FOUND:
    flow_rows.append(("sau cross-dedupe (loại phía VSEC)", len(vsec_final)))
flow_rows += [("train (90%)", len(train_idx)), ("val (10%)", len(val_idx))]
print("== Flow số câu (VSEC) ==")
print(pd.DataFrame(flow_rows, columns=["bước", "số câu"]).to_string(index=False))

train_text_set = {texts[i] for i in train_idx}
val_text_set = {texts[i] for i in val_idx}
inv_sum = len(train_idx) + len(val_idx) == len(vsec_final)
inv_overlap = len(train_text_set & val_text_set)
inv_test_overlap = len((train_text_set | val_text_set) & kept_test_set) if TEST_FOUND else 0
invariants_ok = bool(inv_sum and inv_overlap == 0 and inv_test_overlap == 0)

print()
print(f"Invariant 1 — len(train) + len(val) == sau-dedupe : {inv_sum}")
print(f"Invariant 2 — giao normalized text train ∩ val    : {inv_overlap} (kỳ vọng 0)")
if TEST_FOUND:
    print(f"Invariant 3 — giao VSEC (train+val) ∩ test        : {inv_test_overlap} (kỳ vọng 0)")
print("ALL OK" if invariants_ok else "INVARIANT VIOLATION!")

manifest = {
    "created": RUN_STAMP,
    "notebook": "nb0_data_prep",
    "hf_dataset_id": HF_DATASET_ID,
    "seed": SEED,
    "val_size": VAL_SIZE,
    "neardup_threshold": NEARDUP_THRESHOLD,
    "neardup_scorer": "rapidfuzz.token_set_ratio",
    "cross_dup_keep": CROSS_DUP_KEEP,
    "error_bins": "min(error_count, 4): 1/2/3/>=4",
    "vsec": {
        "raw": len(ds),
        "exact_dup_removed": len(vsec_drop_exact),
        "near_dup_removed": len(vsec_drop_near),
        "cross_dup_removed_exact": cross_drop_exact,
        "cross_dup_removed_near": cross_drop_near,
        "after_dedupe": len(vsec_final),
        "train": len(train_idx),
        "val": len(val_idx),
        "train_error_bins": {bin_names[b]: sum(1 for j in train_pos if bins[j] == b) for b in ERROR_BINS},
        "val_error_bins": {bin_names[b]: sum(1 for j in val_pos if bins[j] == b) for b in ERROR_BINS},
    },
    "test": {
        "found": TEST_FOUND,
        "file": str(test_path) if test_path else None,
        "raw": test_raw_n,
        "exact_dup_removed": test_drop_exact,
        "near_dup_removed": test_drop_near,
        "after_dedupe": len(test_export),
    },
    "annotation_position_mismatch": {
        "rows": n_mismatch_rows,
        "annotations": n_mismatch_ann,
    },
    "invariants_ok": invariants_ok,
}
with open(OUTPUT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print()
print("Files trong output dir:", sorted(p.name for p in OUTPUT_DIR.iterdir()))
print(json.dumps(manifest, ensure_ascii=False, indent=2))

## 9. Đưa output lên Kaggle Dataset cho nb1

Sau khi chạy xong trên Kaggle, output nằm ở `/kaggle/working`: `vsec_train.jsonl`, `vsec_val.jsonl`, `manifest.json` (và `test_normalized.jsonl` nếu có bộ test).

1. Ở trang notebook Kaggle (phiên bản đã **Save Version → Run All**): panel bên phải → **Output** → chọn các file output → **New Dataset** (ví dụ đặt tên `vsec-prep0`). Cách khác: tải file về máy rồi tạo dataset thủ công tại kaggle.com/datasets → **New Dataset**.
2. Trong nb1 (`nb1_align_annotate`): **Add Input** → chọn dataset vừa tạo; file sẽ nằm dưới `/kaggle/input/<tên-dataset>/`.
3. nb1 đọc `manifest.json` trước để lấy seed/ngưỡng/số câu, bảo đảm nhất quán giữa các notebook (nguyên tắc DESIGN.md §7).

Kích thước output (~9k câu text + manifest) rất nhỏ so với giới hạn dataset của Kaggle — không vấn đề.